In [ ]:
# Batch fixer for AWS markdown image placement
from pathlib import Path
import re

ROOT = Path(r"c:\Users\Admin\Desktop\new-learnings\Learnings\AWS")
EXTENSIONS = {".md", ".markdown"}
IMAGE_PATTERN = re.compile(r"!\[[^\]]*\]\([^\)]*\)")


def normalize_image_lines(text: str) -> str:
    lines = text.splitlines()
    output = []
    in_code = False

    for line in lines:
        if line.strip().startswith("```"):
            in_code = not in_code
            output.append(line)
            continue

        if in_code:
            output.append(line)
            continue

        if line.lstrip().startswith("- !["):
            line = line[line.index("!["):]

        if IMAGE_PATTERN.search(line) and line.strip().startswith("!["):
            if output and output[-1].strip() != "":
                output.append("")
        output.append(line)

    return "\n".join(output) + ("\n" if text.endswith("\n") else "")


def process_file(path: Path) -> bool:
    text = path.read_text(encoding="utf-8")
    fixed = normalize_image_lines(text)
    if fixed != text:
        path.write_text(fixed, encoding="utf-8")
        return True
    return False


def run_batch():
    changed_files = []
    for path in ROOT.rglob("*"):
        if path.suffix.lower() in EXTENSIONS:
            if process_file(path):
                changed_files.append(path)
    return changed_files


if __name__ == "__main__":
    changed = run_batch()
    print(f"Changed {len(changed)} files")
    for p in changed:
        print(p)
